# "Chiến dịch ở Đức tăng 68% khách": kiểm tra một claim bằng difference-in-differences

Agency quảng cáo gửi báo cáo cho Giám đốc marketing:

> "Từ **1/9/2011** chúng tôi chạy quảng cáo nhắm khách doanh nghiệp ở **Đức**. Trong 12 tuần sau đó, số khách Đức
> mua hàng mỗi tuần tăng **68%** so với 27 tuần trước (tháng 3-8). Đề nghị tăng gấp đôi ngân sách quý tới."

Giám đốc marketing hỏi bạn: **con số 68% có phải là tác động của chiến dịch không?**

Không có A/B test: mọi khách Đức đều "được nhắm". Cách nhanh nhất để DA kiểm một claim kiểu này là
**difference-in-differences (DiD)**: so thay đổi của nhóm được tác động (Đức) với thay đổi **cùng giai đoạn** của
một nhóm đối chứng không bị tác động (các nước châu Âu khác), để trừ phần thay đổi chung (mùa vụ cuối năm, xu
hướng của cả công ty).

**Kịch bản giả định**: dữ liệu `online-retail-ii` là thật nhưng **không có thông tin về chiến dịch nào**; agency và
chiến dịch là giả định để luyện cách kiểm tra claim. Bạn đang làm đúng việc của DA khi nhận một báo cáo
"trước/sau": tìm phần thay đổi mà chiến dịch không giải thích được.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/advanced/02-did-campaign-claim -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Nhóm đối chứng: các nước châu Âu khác có khách mua thường xuyên (không gồm Anh: thị trường nội địa, quy mô và
# mùa vụ khác hẳn; không gồm "European Community": không rõ nước nào).
TREATED = ["Germany"]
CONTROL = ["France", "Belgium", "Netherlands", "Spain", "Switzerland", "Portugal", "Italy", "Austria", "Denmark",
           "Sweden", "Finland", "Norway", "EIRE", "Poland", "Cyprus", "Channel Islands"]
GROUPS = {"germany": TREATED, "control": CONTROL}


def valid_customer_lines(raw: pd.DataFrame) -> pd.DataFrame:
    """Dòng hợp lệ có mã khách (định nghĩa chung của track): customer_id, invoice_date, country. (Đã viết sẵn.)"""
    valid = raw[~raw["invoice"].astype(str).str.startswith("C") & (raw["quantity"] > 0) & (raw["price"] > 0)
                & raw["customer_id"].notna()]
    return valid[["customer_id", "invoice_date", "country"]].reset_index(drop=True)

## Phần A. Panel theo tuần

Metric: **số khách mua hàng mỗi tuần** (khách có ít nhất một dòng hợp lệ trong tuần), đúng metric của agency.

`weekly_active(lines, groups, start, end) -> DataFrame`:

- chỉ lấy dòng có `invoice_date` trong `[start, end)`;
- tuần thứ Hai-Chủ nhật: `invoice_date.dt.to_period("W-SUN")`;
- mỗi nhóm trong `groups` (dict `tên nhóm -> danh sách country`) một cột, theo đúng thứ tự khoá của dict; giá trị
  là số `customer_id` **khác nhau** của các nước trong nhóm trong tuần đó; nước không thuộc nhóm nào bị bỏ;
- index là các tuần có ít nhất một dòng (của bất kỳ nhóm nào), sắp tăng dần, tên `week`; nhóm không có khách
  trong một tuần thì là `0`; kiểu số nguyên.

In [ ]:
def weekly_active(lines: pd.DataFrame, groups: dict, start, end) -> pd.DataFrame:
    """Số khách khác nhau mỗi tuần (W-SUN) của từng nhóm nước, dòng trong [start, end)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: nạp Online Retail II (lần đầu ~8 giây, có cache parquet trong thư mục tạm)
import sys
import os
import tempfile
import uuid
from pathlib import Path

import duckdb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

SHEETS = ("Year 2009-2010", "Year 2010-2011")  # sheet đầu chứa lại 01-09/12/2010 của sheet sau: bỏ phần trùng
CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache parquet dùng chung với module 03, không ghi vào data/raw


def _sql_path(path: Path) -> str:
    return str(path).replace("'", "''")


def _sheet_sql(xlsx: Path, sheet: str) -> str:
    return f"""
        SELECT Invoice AS invoice, StockCode AS stock_code, Description AS description,
               CAST(CAST(Quantity AS DOUBLE) AS INTEGER) AS quantity,
               TIMESTAMP '1899-12-30' + to_seconds(CAST(round(CAST(InvoiceDate AS DOUBLE) * 86400) AS BIGINT))
                   AS invoice_date,
               CAST(Price AS DOUBLE) AS price,
               CAST(CAST("Customer ID" AS DOUBLE) AS INTEGER) AS customer_id,
               Country AS country
        FROM read_xlsx('{_sql_path(xlsx)}', sheet = '{sheet}', all_varchar = true)"""


def _excel_extension_missing(exc: duckdb.Error) -> bool:
    """Chỉ lỗi cài/nạp extension excel (máy offline...). Lỗi SQL hay lỗi dữ liệu thì phải nổi lên."""
    kinds = (duckdb.CatalogException, duckdb.IOException, duckdb.HTTPException)
    return isinstance(exc, kinds) and "extension" in str(exc).lower()


def load_online_retail(xlsx: Path) -> pd.DataFrame:
    """Hai sheet Online Retail II gộp (chưa lọc), cột snake_case: invoice, stock_code, description, quantity,
    invoice_date, price, customer_id, country.

    Lần đầu đọc xlsx bằng extension excel của DuckDB (~8 giây; máy không cài được extension thì pandas ~60-90 giây)
    rồi ghi cache parquet vào thư mục tạm của hệ điều hành, khoá theo mtime và kích thước tệp; lần sau ~1 giây.
    """
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_ii_{stat.st_mtime_ns}_{stat.st_size}.parquet"
    con = duckdb.connect()
    if cache.exists():
        try:
            return con.execute(f"SELECT * FROM read_parquet('{_sql_path(cache)}')").df()
        except duckdb.IOException:  # cache hỏng: đọc lại từ xlsx và ghi đè
            pass
    try:
        con.execute(
            f"CREATE TABLE online_retail AS SELECT * FROM ({_sheet_sql(xlsx, SHEETS[0])}) "
            f"WHERE invoice_date < TIMESTAMP '2010-12-01' UNION ALL {_sheet_sql(xlsx, SHEETS[1])}"
        )
    except duckdb.Error as exc:
        if not _excel_extension_missing(exc):
            raise
        old, new = (pd.read_excel(xlsx, sheet_name=s) for s in SHEETS)
        raw = pd.concat([old[old.InvoiceDate < "2010-12-01"], new], ignore_index=True)
        frame = pd.DataFrame({
            "invoice": raw["Invoice"].astype(str), "stock_code": raw["StockCode"].astype(str),
            "description": raw["Description"], "quantity": raw["Quantity"].astype("int32"),
            "invoice_date": raw["InvoiceDate"], "price": raw["Price"].astype(float),
            "customer_id": raw["Customer ID"].astype("Int32"), "country": raw["Country"],
        })
        con.register("frame", frame)
        con.execute("CREATE TABLE online_retail AS SELECT * FROM frame")
    tmp = CACHE_DIR / f".{cache.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    try:  # ghi atomic: tệp tạm tên riêng rồi rename, an toàn khi nhiều tiến trình cùng ghi
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        con.execute(f"COPY online_retail TO '{_sql_path(tmp)}' (FORMAT parquet)")
        os.replace(tmp, cache)
    except (OSError, duckdb.Error):  # không ghi được cache thì thôi, lần sau đọc lại xlsx
        tmp.unlink(missing_ok=True)
    return con.execute("SELECT * FROM online_retail").df()


raw = load_online_retail(ensure("online-retail-ii") / "online_retail_II.xlsx")
lines = valid_customer_lines(raw)
print(f"{len(lines):,} dòng hợp lệ có mã khách")

In [ ]:
# explore: panel 39 tuần (28/2/2011 - 27/11/2011), tái tạo con số của agency
CUTOFF = "2011-09-01"
weekly = weekly_active(lines, GROUPS, "2011-02-28", "2011-11-28")
post = weekly.index.start_time >= pd.Timestamp(CUTOFF)
print(f"{len(weekly)} tuần: {(~post).sum()} trước, {post.sum()} sau (tuần chứa 1/9 bắt đầu 29/8 nên tính là trước)")
print(weekly.describe().round(1).to_string())
de = weekly["germany"]
print(f"\nĐức: {de[~post].mean():.2f} -> {de[post].mean():.2f} khách/tuần, {de[post].mean() / de[~post].mean() - 1:+.1%} (con số của agency)")

fig, ax = plt.subplots(figsize=(10, 3.5))
for name, color in [("germany", "#e45756"), ("control", "#4c78a8")]:
    s = weekly[name] / weekly[name][~post].mean() * 100  # chỉ số hoá: TB giai đoạn trước = 100
    ax.plot(weekly.index.start_time, s, color=color, label=name)
ax.axvline(pd.Timestamp(CUTOFF), color="#5f6b7a", ls="--", lw=1)
ax.annotate("chiến dịch (giả định)", (pd.Timestamp(CUTOFF), 250), color="#5f6b7a")
ax.set_ylabel("khách/tuần, TB tháng 3-8 = 100")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Khách Đức và khách các nước châu Âu khác cùng tăng mạnh từ tháng 9", loc="left")
plt.show()

## Phần B. Ước lượng DiD

`did_table(weekly, treated, control, cutoff) -> dict`: tuần **sau** là tuần có `start_time >= cutoff`, còn lại là
**trước**. Trả dict với các khoá:

- `pre_treated`, `post_treated`, `pre_control`, `post_control`: trung bình khách/tuần của mỗi nhóm mỗi giai đoạn;
- `change_treated`, `change_control`: thay đổi tương đối (`post / pre - 1`);
- `did_log` = `log(post_treated / pre_treated) - log(post_control / pre_control)`;
- `did` = `exp(did_log) - 1`: nhóm được tác động tăng **thêm** bao nhiêu phần trăm so với mức nhóm đối chứng tăng.

Vì sao dùng log (thay đổi tương đối) chứ không trừ số tuyệt đối: Đức có khoảng 6 khách/tuần, nhóm đối chứng
khoảng 17. Giả định **xu hướng song song** hợp lý hơn trên thang tỉ lệ (mùa Giáng sinh làm mọi nước tăng cùng một
tỉ lệ, không cùng một số khách).

`did_regression(weekly, treated, control, cutoff, maxlags=4) -> dict`: cùng ước lượng đó nhưng có khoảng tin cậy.
Dựng `gap = log(weekly[treated]) - log(weekly[control])` mỗi tuần và `post` (1 nếu tuần sau `cutoff`, 0 nếu
trước), rồi `smf.ols("gap ~ post", data).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})`. Hệ số của `post`
chính là DiD trên log (chênh lệch trung bình của khoảng cách log giữa hai giai đoạn); HAC vì các tuần liền nhau
tương quan với nhau. Trả dict `effect_log`, `ci_low`, `ci_high` (khoảng tin cậy 95% trên thang log), `p`,
`effect` (`exp(effect_log) - 1`). Tuần có 0 khách ở một nhóm thì log không xác định: dữ liệu ở đây không có, nhưng
hãy nghĩ cách xử lý nếu có.

In [ ]:
def did_table(weekly: pd.DataFrame, treated: str, control: str, cutoff) -> dict:
    """DiD 2x2 trên trung bình khách/tuần, thang log."""
    # TODO
    raise NotImplementedError


def did_regression(weekly: pd.DataFrame, treated: str, control: str, cutoff, maxlags: int = 4) -> dict:
    """OLS gap ~ post với sai số chuẩn HAC: effect_log, ci_low, ci_high, p, effect."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: DiD 2x2 và hồi quy
table = did_table(weekly, "germany", "control", CUTOFF)
print({k: round(float(v), 4) for k, v in table.items()})
reg = did_regression(weekly, "germany", "control", CUTOFF)
print({k: round(v, 4) for k, v in reg.items()})
print(f"DiD: {reg['effect']:+.1%}, CI 95% [{np.exp(reg['ci_low']) - 1:+.1%}; {np.exp(reg['ci_high']) - 1:+.1%}], p = {reg['p']:.2f}")

gap = np.log(weekly["germany"]) - np.log(weekly["control"])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(weekly.index.start_time, gap, color="#5f6b7a", marker="o", ms=3, lw=1)
for mask, label in [(~post, "TB trước"), (post, "TB sau")]:
    ax.hlines(gap[mask].mean(), weekly.index.start_time[mask].min(), weekly.index.start_time[mask].max(),
              color="#e45756", lw=2)
ax.axvline(pd.Timestamp(CUTOFF), color="#5f6b7a", ls="--", lw=1)
ax.set_ylabel("log(Đức) - log(đối chứng)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Khoảng cách Đức - đối chứng dao động mạnh từ tuần này sang tuần khác", loc="left")
plt.show()

In [ ]:
# explore: placebo. Áp đúng quy trình vào nơi CHẮC CHẮN không có chiến dịch: con số ra lớn cỡ nào?
placebo_time = weekly_active(lines, GROUPS, "2011-02-28", "2011-08-29")      # chỉ giai đoạn trước, mốc giả 1/6
placebo_year = weekly_active(lines, GROUPS, "2010-03-01", "2010-11-29")      # cùng mùa năm trước, mốc giả 1/9/2010
for label, panel, cut in [("mốc giả 1/6/2011", placebo_time, "2011-06-01"),
                          ("cùng mùa 2010", placebo_year, "2010-09-01")]:
    t, r = did_table(panel, "germany", "control", cut), did_regression(panel, "germany", "control", cut)
    print(f"{label}: Đức {t['change_treated']:+.1%}, đối chứng {t['change_control']:+.1%}, DiD {t['did']:+.1%}; "
          f"hồi quy {r['effect']:+.1%} (p = {r['p']:.2f})")

# explore: xu hướng trước có song song không? Độ dốc của gap trong giai đoạn trước
pre = pd.DataFrame({"gap": gap[~post].to_numpy(), "t": np.arange((~post).sum())})
slope = smf.ols("gap ~ t", pre).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(f"độ dốc gap trước chiến dịch: {slope.params['t']:+.4f}/tuần (p = {slope.pvalues['t']:.2f})")

## Viết MEMO.md

Mở `starter/MEMO.md` và viết memo cho Giám đốc marketing (200-600 từ, đủ các mục, dòng đầu là headline có con
số, xoá mọi dòng TODO). Câu hỏi để tự kiểm trước khi viết:

1. Con số 68% của agency đúng về số học không? Nó so cái gì với cái gì, và thiếu cái gì?
2. Cùng giai đoạn, các nước châu Âu khác tăng bao nhiêu? Năm 2010 (không có chiến dịch) Đức tăng bao nhiêu từ
   tháng 3-8 sang tháng 9-11?
3. DiD ước lượng bao nhiêu, khoảng tin cậy ra sao? Placebo cho thấy "nhiễu" của phương pháp lớn cỡ nào? Với
   khoảng 6 khách Đức mỗi tuần, phương pháp này phát hiện được hiệu ứng nhỏ cỡ nào?
4. Giả định xu hướng song song có đứng được không? Điều gì có thể làm DiD sai (một khách sỉ Đức đổi lịch đặt
   hàng, chiến dịch "tràn" sang Áo/Thuỵ Sĩ, khủng hoảng nợ công châu Âu 2011 ảnh hưởng khác nhau theo nước)?
5. Bạn đề xuất gì cho ngân sách quý tới: tăng gấp đôi, giữ, hay chạy thế nào để lần sau đo được (geo
   experiment: chọn ngẫu nhiên nước/vùng được chạy, holdout)?